# CMSC 173 &middot; Module 12 &middot; Example 3
## Handwritten digits

1,797 handwritten digits, 64 pixels each. One hidden layer of 32 ReLU neurons, a softmax output of 10; then the network's seven mistakes, and overfitting watched on a small training set.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (scikit-learn 1.4.2; a newer version can differ slightly for the trained networks);
change something and see what moves.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = load_digits(return_X_y=True)                # 1,797 images, 64 pixels each
X_tr, X_te, y_tr, y_te = train_test_split(X / 16, y, test_size=0.3, random_state=173)
net = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", alpha=1e-4, learning_rate_init=0.001,
                    batch_size=64, max_iter=300, random_state=173).fit(X_tr, y_tr)
print(X_tr.shape, net.n_iter_, round(net.loss_, 4))                         # (1257, 64) 291 0.004
print(round(net.score(X_te, y_te), 3), (net.predict(X_te) != y_te).sum(), "mistakes")   # 0.987 7 mistakes
print(sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_), "parameters")   # 2410

**The seven mistakes.**

In [ ]:
import matplotlib.pyplot as plt

pred = net.predict(X_te)
wrong = np.where(pred != y_te)[0]
fig, axes = plt.subplots(1, len(wrong), figsize=(14, 2.4))
for ax, i in zip(axes, wrong):
    ax.imshow(X_te[i].reshape(8, 8), cmap="Greys")
    ax.set_title(f"a {y_te[i]}, read as {pred[i]}"); ax.axis("off")
plt.show()

**Compared with module 6's logistic regression** on the same split.

In [ ]:
from sklearn.linear_model import LogisticRegression
print(round(LogisticRegression(max_iter=5000).fit(X_tr, y_tr).score(X_te, y_te), 3))   # 0.972

**Overfitting, watched.** 200 training digits and 256 hidden neurons: far more weights than examples. This cell trains two networks for 150 epochs each (about a minute).

In [ ]:
from sklearn.metrics import log_loss

Xs_tr, Xs_va, ys_tr, ys_va = train_test_split(X / 16, y, train_size=200, random_state=173, stratify=y)
for alpha in (0.0, 0.03):
    net = MLPClassifier((256,), alpha=alpha, learning_rate_init=0.005, batch_size=32, random_state=173)
    val = []
    for epoch in range(150):
        net.partial_fit(Xs_tr, ys_tr, classes=np.arange(10))
        val.append(log_loss(ys_va, net.predict_proba(Xs_va), labels=np.arange(10)))
    plt.plot(val, label=f"alpha = {alpha}")
    print(alpha, "lowest", round(min(val), 3), "at epoch", int(np.argmin(val)) + 1, "| epoch 150:", round(val[-1], 3))
# 0.0: lowest 0.25 at epoch 19, then up to 0.321 | 0.03: lowest 0.244 at epoch 37, 0.258 at 150
plt.xlabel("epoch"); plt.ylabel("validation log loss"); plt.ylim(0, 0.8); plt.legend(); plt.show()

**Try it.** Change `hidden_layer_sizes=(32,)` to `(8,)` and to `(128,)`. How many mistakes now?